# Day 11 — GNN Training → GCN：补完上一讲，再正式进入 GCN

## Day 11 两段主线

### Part A — 补完 GNN1
1. GNN 可学习参数到底是什么
2. Matrix view 回顾
3. 如何训练 GNN
4. supervised vs unsupervised objective
5. end-to-end backprop
6. shared parameters 与 inductive generalization

### Part B — 正式进入 GCN
7. General GNN = Message + Aggregation
8. GCN 的 Message / Aggregation
9. self-loop
10. degree normalization
11. symmetric normalization
12. 手搓 `SimpleGCNConv`
13. PyG `GCNConv`
14. semi-supervised node classification

# Part A — 补完上一讲：GNN 到底怎么训练？

## 1. 从 Day10 的 forward 接上 training

Day10 已经得到：

$$
h_v^{(k+1)}
=
\sigma\left(
W_k
\frac{1}{|N(v)|}
\sum_{u\in N(v)}h_u^{(k)}
+
B_k h_v^{(k)}
\right)
$$

其中：

- $W_k$：neighbor message 的参数
- $B_k$：self transformation 的参数
- $h_v^{(0)}=x_v$
- $z_v=h_v^{(K)}$

Day10 主要回答了：

> **forward 时 representation 怎么从一层传到下一层？**

现在补上：

> **这些 $W_k,B_k$ 到底怎么学出来？**

## 2. 可学习参数 vs 中间表示

非常重要：

$$
W_k,\;B_k
$$

是 **trainable parameters**。

而：

$$
H^{(k)},\;h_v^{(k)},\;z_v
$$

通常是 forward 过程中计算得到的 **representations / activations**，不是 optimizer 直接维护的一张 embedding table。

所以 GNN 与 Day9 shallow embedding 的核心区别之一是：

```text
shallow embedding:
node id → 每个 node 自己一行可学习参数

GNN:
node feature + graph
→ 共享 W/B
→ 算出 node representation
```

### 必须回答 A1

为什么 $h_v^{(k)}$ 会随着训练变化，但它本身通常不是 optimizer 直接更新的 parameter？

你的回答：$h_v^{(k)}$ 是由输入特征、图结构以及当前层参数 W,B 前向计算得到的中间表示，不是独立的可学习参数。optimizer 更新的是 W,B 等模型参数；参数改变后，下一次 forward 得到的 $h_v^{(k)}$ 也会随之改变。

## 3. Matrix View 回顾：这一块 Day10 已经学过

把所有 node representation 堆成：

$$
H^{(k)}\in\mathbb{R}^{|V|\times d}
$$

邻居求和：

$$
AH^{(k)}
$$

邻居 mean：

$$
D^{-1}AH^{(k)}
$$

基础 update 可写成：

$$
H^{(k+1)}
=
\sigma\left(
D^{-1}AH^{(k)}W_k^T
+
H^{(k)}B_k^T
\right)
$$

这一节只承上启下，不重复手搓。

## 4. 如何训练 GNN：最终一定要先定义 Loss

得到最终 node embedding：

$$
z_v=h_v^{(K)}
$$

之后，还不能训练。

必须先定义：

$$
\mathcal L
$$

然后：

```text
Graph + X
↓
GNN forward
↓
z_v
↓
task head / decoder
↓
loss
↓
backward
↓
gradients of W/B/head
↓
optimizer.step()
```

这就是 **end-to-end training**。

## 5. Supervised setting

如果有 node label $y_v$：

$$
z_v=f_\Theta(G,X,v)
$$

再接 classifier：

$$
\hat y_v=g_\phi(z_v)
$$

然后：

$$
\min_{\Theta,\phi}
\mathcal L(y_v,\hat y_v)
$$

如果是 categorical classification，常用 Cross Entropy。

例如 $C$ 类：

$$
\text{logits}_v = z_vW_c
$$

$$
\mathcal L
=
\operatorname{CE}(\text{logits}_v,y_v)
$$

## 6. Unsupervised setting

如果没有 node label：

> **用 graph structure 本身构造 supervision。**

例如先定义 node similarity：

- random-walk co-occurrence
- edge connectivity
- matrix-factorization-derived similarity

然后希望 similar nodes 的 embeddings 更匹配。

一个典型思路：

$$
s(u,v)=z_u^Tz_v
$$

再用 positive / negative pairs 构造 loss。

这和 Day9 的 random walk + negative sampling 思路是连起来的；区别只是：

```text
Day9 encoder = embedding lookup
现在 encoder = GNN
```

### 必须回答 A2

为什么说 supervised / unsupervised 的主要区别，不在 GNN encoder 本身，而在 **supervision signal / objective 从哪里来**？

你的回答：supervised 和 unsupervised 的核心区别，是 supervision signal 从哪里来。
supervised 直接用人工/真实标签 \(y\)；unsupervised 没有显式标签，就从图结构、random walk、node similarity 等构造训练信号。
但它们都可以用同一个 GNN encoder，forward/backward/optimizer 这套训练机制本身并没有变

## 7. End-to-End Backprop：把链条补完整

两层 GNN：

$$
H^{(1)}=\operatorname{MP}_1(G,H^{(0)})
$$

$$
H^{(2)}=\operatorname{MP}_2(G,H^{(1)})
$$

最后：

$$
\text{logits}=H^{(2)}W_c
$$

$$
\mathcal L=\operatorname{CE}(\text{logits},y)
$$

调用一次：

```python
loss.backward()
```

Autograd 会按 chain rule：

```text
loss
↓
classifier
↓
H2
↓
layer2 W/B
↓
H1
↓
layer1 W/B
```

所以最终任务 loss 能直接训练最前面的 message-passing 参数。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import random

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

G = nx.Graph()
G.add_edges_from([
    (0, 1), (0, 2),
    (1, 3), (1, 4),
    (2, 5)
])

X = torch.tensor([
    [1.0, 0.0, 0.5],
    [0.5, 1.0, 0.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0],
    [0.0, 0.5, 1.0],
    [1.0, 0.0, 1.0],
])

In [2]:
def mean_aggregate_one_node(G, H, v):
    nbrs = list(G.neighbors(v))
    if len(nbrs) == 0:
        return torch.zeros(H.shape[1], dtype=H.dtype)
    return H[nbrs].mean(dim=0)


class ToyMeanMessagePassing(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W_nbr = nn.Linear(in_dim, out_dim, bias=False)
        self.W_self = nn.Linear(in_dim, out_dim, bias=True)

    def forward(self, G, H):
        out = []

        for v in G.nodes():
            m_v = mean_aggregate_one_node(G, H, v)
            h_new = self.W_nbr(m_v) + self.W_self(H[v])
            h_new = F.relu(h_new)
            out.append(h_new)

        return torch.stack(out)

In [3]:
layer1 = ToyMeanMessagePassing(3, 4)
layer2 = ToyMeanMessagePassing(4, 2)

y = torch.tensor([0, 0, 1, 1, 1, 0])

optimizer = torch.optim.Adam(
    list(layer1.parameters()) + list(layer2.parameters()),
    lr = 1e-2
)

optimizer.zero_grad()

H1 = layer1(G, X)
logits = layer2(G, H1)
loss = F.cross_entropy(logits, y)

loss.backward()

print("loss =", loss.item())
print("layer1 W_nbr grad exists?",
      layer1.W_nbr.weight.grad is not None)
print("layer1 grad norm =",
      layer1.W_nbr.weight.grad.norm().item())

optimizer.step()

loss = 0.7874720692634583
layer1 W_nbr grad exists? True
layer1 grad norm = 0.18398232758045197


### 必须回答 A3

上面的 `loss.backward()` 为什么会让 `layer1.W_nbr.weight.grad` 非空？

你的回答：一次 loss.backward() 会沿计算图把梯度传回所有参与当前 loss 计算的可学习参数，并把梯度写到它们的 .grad 中；真正更新参数是在后面的 optimizer.step()

## 8. Shared parameters → Inductive potential

GNN 的同一层对所有 node 共享同一组参数：

$$
W_k,\;B_k
$$

因此参数数量不需要随着 node 数量线性增加。

如果 test time 出现一个新 node，只要有：

- node feature
- neighborhood / graph connectivity

就可以继续使用同一套 message-passing rule 计算它的 representation。

这就是 GNN 相比 shallow embedding 更强的 **inductive potential**。

# Part B — 正式进入 GCN

## 9. 用新版课件统一语言：GNN Layer = Message + Aggregation

一个 GNN layer 可以抽象成两步。

### Message

$$
m_u^{(l)}
=
\operatorname{MSG}^{(l)}
\left(h_u^{(l-1)}\right)
$$

例如：

$$
m_u^{(l)}=W^{(l)}h_u^{(l-1)}
$$

### Aggregation

$$
h_v^{(l)}
=
\operatorname{AGG}^{(l)}
\left(
\{m_u^{(l)}:u\in N(v)\}
\right)
$$

AGG 可以是：

- Sum
- Mean
- Max

Day10 的 `AGGREGATE + UPDATE` 与这里没有矛盾；只是现在用更一般的框架重新组织。

### 必须回答 B1

Message 和 Aggregation 各自解决什么问题？

你的回答：
Message：对每个节点当前的 representation 做变换，得到它要传递给其他节点的信息，例如 $m_u=W h_u$。
Aggregation：目标节点把所有邻居发来的 messages 用 sum / mean / max 等顺序不敏感的方式聚合起来，得到新的 neighborhood representation。

## 10. Classical GCN：它在这个框架里是什么？

课件先给一个直观版本：

$$
h_v^{(l)}
=
\sigma\left(
W^{(l)}
\frac{1}{|N(v)|}
\sum_{u\in N(v)}
h_u^{(l-1)}
\right)
$$

也就是：

> **对 neighborhood 做 degree-normalized mean，再做共享线性变换和 activation。**

在 GCN 里，通常把 self-edge 也包含在 graph 中，所以 $N(v)$ 可以包含 node $v$ 自己。

## 11. Self-loop：$\tilde A=A+I$

普通 adjacency matrix 常有：

$$
A_{vv}=0
$$

如果希望自己也参与 aggregation，可以：

$$
\tilde A=A+I
$$

于是：

$$
\tilde A_{vv}=1
$$

这就是给每个 node 加 self-loop。

In [4]:
A = torch.tensor(nx.to_numpy_array(G), dtype=torch.float32)
I = torch.eye(A.shape[0])
A_tilde = A + I

print("A =\n", A)
print("\nA_tilde =\n", A_tilde)

A =
 tensor([[0., 1., 1., 0., 0., 0.],
        [1., 0., 0., 1., 1., 0.],
        [1., 0., 0., 0., 0., 1.],
        [0., 1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.]])

A_tilde =
 tensor([[1., 1., 1., 0., 0., 0.],
        [1., 1., 0., 1., 1., 0.],
        [1., 0., 1., 0., 0., 1.],
        [0., 1., 0., 1., 0., 0.],
        [0., 1., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0., 1.]])


### 必须回答 B2

为什么 $A+I$ 就会让 node 自己的 representation 进入 aggregation？

你的回答：因为 $A+I$ 会让邻接矩阵对角线元素变成 1，也就是给每个 node 加一个 self-loop。这样在计算 $\tilde A H_v=\sum_u \tilde A_{vu}h_u$ 时，$u=v$ 这一项会保留 $h_v$，所以 node 自己的 representation 也进入 aggregation。

## 12. Row normalization：先和课件的 mean 对上

定义加入 self-loop 后的 degree：

$$
\tilde D_{vv}
=
\sum_u\tilde A_{vu}
$$

那么：

$$
\tilde D^{-1}\tilde A H
$$

就是每个 node 的：

> **self + neighbors mean aggregation**

In [6]:
degree = A_tilde.sum(dim=1, keepdim=True)
A_mean = A_tilde / degree

neighbor_mean = A_mean @ X

print("degree =", degree.squeeze())
print("neighbor mean shape =", neighbor_mean.shape)
print(neighbor_mean)

degree = tensor([3., 4., 3., 2., 2., 2.])
neighbor mean shape = torch.Size([6, 3])
tensor([[0.5000, 0.6667, 0.5000],
        [0.6250, 0.6250, 0.3750],
        [0.6667, 0.3333, 0.8333],
        [0.7500, 1.0000, 0.0000],
        [0.2500, 0.7500, 0.5000],
        [0.5000, 0.5000, 1.0000]])


## 13. 原始 GCN / PyG 常见的 symmetric normalization

课件提醒：原始 GCN paper 使用稍有不同的 normalization。

经典形式：

$$
\hat A
=
\tilde D^{-1/2}
\tilde A
\tilde D^{-1/2}
$$

于是：

$$
H^{(l+1)}
=
\sigma\left(
\hat A H^{(l)}W^{(l)}
\right)
$$

对一条 $u\to v$：

$$
\hat A_{vu}
=
\frac{\tilde A_{vu}}
{\sqrt{\tilde d_v}\sqrt{\tilde d_u}}
$$

因此 sender 和 receiver 的 degree 都参与尺度修正。

In [8]:
degree_vec = A_tilde.sum(dim=1)
deg_inv_sqrt = degree_vec.pow(-0.5)
D_inv_sqrt = torch.diag(deg_inv_sqrt)

A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt

print("A_hat =\n", A_hat)

A_hat =
 tensor([[0.3333, 0.2887, 0.3333, 0.0000, 0.0000, 0.0000],
        [0.2887, 0.2500, 0.0000, 0.3536, 0.3536, 0.0000],
        [0.3333, 0.0000, 0.3333, 0.0000, 0.0000, 0.4082],
        [0.0000, 0.3536, 0.0000, 0.5000, 0.0000, 0.0000],
        [0.0000, 0.3536, 0.0000, 0.0000, 0.5000, 0.0000],
        [0.0000, 0.0000, 0.4082, 0.0000, 0.0000, 0.5000]])


### 必须回答 B3

直觉上：

$$
\tilde D^{-1}\tilde A
$$

和

$$
\tilde D^{-1/2}\tilde A\tilde D^{-1/2}
$$

有什么区别？

你的回答：$\tilde D^{-1}\tilde A$ 只根据目标节点 v 的 degree 做 row normalization，相当于对 node v 收到的邻居信息取平均，而$\tilde D^{-1/2}\tilde A\tilde D^{-1/2}$ 则对每条边同时根据 sender u 和 receiver v 的 degree 做对称缩放。

## 14. 手搓 `SimpleGCNConv`

这一段是 Day11 的核心代码交付。

In [ ]:
def normalize_adjacency(A):
    n = A.shape[0]

    A_tilde = A + torch.eye(
        n, dtype=A.dtype, device=A.device
    )

    degree = A_tilde.sum(dim=1)
    deg_inv_sqrt = degree.pow(-0.5)
    D_inv_sqrt = torch.diag(deg_inv_sqrt)

    return D_inv_sqrt @ A_tilde @ D_inv_sqrt


class SimpleGCNConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, A, H):
        A_hat = normalize_adjacency(A)
        H_transformed = self.linear(H)
        return A_hat @ H_transformed

In [11]:
def normalize_adjacency(A):
    n = A.shape[0]

    A_tilde = A + torch.eye(
        n, dtype=A.dtype, device=A.device
    )

    degree = A_tilde.sum(dim=1)
    deg_inv_sqrt = degree.pow(-0.5)
    D_inv_sqrt = torch.diag(deg_inv_sqrt)
    return D_inv_sqrt @ A_tilde @ D_inv_sqrt

class SimpleGCNConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, A, H):
        A_hat = normalize_adjacency(A)
        H_transformed = self.linear(H)
        return A_hat @ H_transformed

In [12]:
gcn1 = SimpleGCNConv(3,4)
gcn2 = SimpleGCNConv(4,2)

H1 = F.relu(gcn1(A, X))
H2 = gcn2(A, H1)

print("H0:", X.shape)
print("H1:", H1.shape)
print("H2:", H2.shape)
print(H2)

H0: torch.Size([6, 3])
H1: torch.Size([6, 4])
H2: torch.Size([6, 2])
tensor([[0.0940, 0.0568],
        [0.0708, 0.0677],
        [0.1218, 0.0697],
        [0.0381, 0.0362],
        [0.0611, 0.0558],
        [0.1189, 0.0612]], grad_fn=<MmBackward0>)


### 必须回答 B4

为什么：

```python
A_hat @ H_transformed
```

可以一次性完成所有 node 的 normalized message passing？

你的回答：$AH$ 的第 $v$ 行会自动把与 $v$ 相连的所有 node representation 按 $\hat A_{vu}$ 权重加权求和，因此一整个矩阵乘法同时完成所有节点的 normalized aggregation。

## 15. PyG `GCNConv`

真实 graph 往往很 sparse，因此不会真的构造巨大 dense adjacency matrix。

PyG 使用：

```text
edge_index
```

来表示 graph connectivity，再由 `GCNConv` 内部完成 sparse message passing 和 normalization。

In [13]:
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv

dataset = KarateClub()
data = dataset[0]

print(data)
print("x:", data.x.shape)
print("edge_index:", data.edge_index.shape)
print("y:", data.y.shape)
print("train nodes:", int(data.train_mask.sum()))

D:\anaconda\envs\graph-learning\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Data(x=[34, 34], edge_index=[2, 156], y=[34], train_mask=[34])
x: torch.Size([34, 34])
edge_index: torch.Size([2, 156])
y: torch.Size([34])
train nodes: 4


In [19]:
class PyGGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

model = PyGGCN(
    dataset.num_features,
    16,
    dataset.num_classes
)

print(model)

PyGGCN(
  (conv1): GCNConv(34, 16)
  (conv2): GCNConv(16, 4)
)


## 16. Semi-supervised Node Classification

训练时只在：

```python
data.train_mask
```

标记的 node 上计算 supervised loss：

$$
\mathcal L
=
\operatorname{CE}
\left(
\text{logits}_{train},
y_{train}
\right)
$$

但 forward 时整张 graph 的：

- node features
- edge structure

仍然参与 message passing。

所以没有 label 的 node 仍可能通过 graph structure 影响 train node 的 representation。

In [20]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

for epoch in range(201):
    model.train()
    optimizer.zero_grad()

    logits = model(data.x, data.edge_index)

    loss = F.cross_entropy(
        logits[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()
    optimizer.step()

    if epoch % 40 == 0 or epoch == 200:
        model.eval()
        with torch.no_grad():
            eval_logits = model(data.x, data.edge_index)
            pred = eval_logits.argmax(dim=1)
            acc = (pred == data.y).float().mean().item()

        print(
            f"epoch={epoch:3d} "
            f"loss={loss.item():.4f} "
            f"all-node acc={acc:.4f}"
        )

epoch=  0 loss=1.3766 all-node acc=0.3529
epoch= 40 loss=0.3079 all-node acc=0.8235
epoch= 80 loss=0.0286 all-node acc=0.8235
epoch=120 loss=0.0151 all-node acc=0.8235
epoch=160 loss=0.0119 all-node acc=0.7941
epoch=200 loss=0.0104 all-node acc=0.7941


### 必须回答 B5

为什么 loss 只在 `train_mask` 上算，但没有 label 的 node 仍然能影响模型训练？

你的回答：没 label 的 node 不参与 loss，但参与 forward message passing，所以仍然会影响有 label node 的表示

# Day11 最终自测

## Part A：补完 GNN training
1. GNN 中真正 trainable 的参数是什么？
2. $H^{(k)}$ 为什么不是 optimizer 直接维护的参数表？
3. supervised 和 unsupervised objective 的监督信号分别来自哪里？
4. end-to-end training 是什么意思？
5. 为什么一次 `loss.backward()` 能把 gradient 传到第一层？
6. shared parameters 为什么带来 inductive potential？

## Part B：GCN
7. GNN Layer = Message + Aggregation 怎么理解？
8. GCN 的 message 和 aggregation 分别是什么？
9. 为什么需要 self-loop？
10. row normalization 对应什么直觉？
11. symmetric normalization 为什么看两端 degree？
12. 两层 GCN 为什么拿到 2-hop 信息？
13. PyG 为什么用 `edge_index` 而不是 dense $A$？
14. `train_mask` 在 semi-supervised node classification 里做什么？

完成后进入 **Day12：GraphSAGE**。